# Part 2 - Task 3: Train a detection model

Using Ultralytics YOLOv8 since the labels are already in YOLO format. This notebook
reuses the patient-level split saved in `dataset/splits.json` (see
`part2_dataset_inspection.ipynb`).

## Arrange the data into YOLO's expected layout

YOLOv8 expects images and labels under `images/{train,val,test}` and
`labels/{train,val,test}`, with each image's label file living in the mirrored
location under `labels/`. Files are copied (not moved) from `dataset/images` and
`dataset/labels` into a new `dataset/yolo/` directory, routed by the patient ->
split mapping.

In [1]:
import json
import shutil
from pathlib import Path

DATASET_DIR = Path("../dataset")
YOLO_DIR = DATASET_DIR / "yolo"

split_map = json.load(open(DATASET_DIR / "splits.json"))

for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

image_files = sorted((DATASET_DIR / "images").glob("*.jpg"))
counts = {"train": 0, "val": 0, "test": 0}

for img_path in image_files:
    patient_id = img_path.stem.split("_")[0]
    split = split_map[patient_id]
    label_path = DATASET_DIR / "labels" / f"{img_path.stem}.txt"

    shutil.copy2(img_path, YOLO_DIR / "images" / split / img_path.name)
    shutil.copy2(label_path, YOLO_DIR / "labels" / split / label_path.name)
    counts[split] += 1

print(f"Copied files into {YOLO_DIR.resolve()}")
print(counts)

Copied files into E:\Bone Union Detection\dataset\yolo
{'train': 574, 'val': 123, 'test': 123}


## Dataset config

A `data.yaml` file tells YOLOv8 where the splits live and names the single class.

In [2]:
data_yaml = f"""path: {YOLO_DIR.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site
"""

data_yaml_path = YOLO_DIR / "data.yaml"
data_yaml_path.write_text(data_yaml)
print(data_yaml_path.read_text())

path: E:/Bone Union Detection/dataset/yolo
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site



## Train

**Note on scope.** This machine has no GPU (CPU-only PyTorch). A timed 1-epoch run
measured ~122 seconds/epoch; a full training schedule (on the order of 100 epochs
with early stopping, as would normally be used here) would take on the order of
several hours. Given the time available for this submission, a full training run was
not completed. The cell below instead runs a short 1-epoch proof-of-concept to
demonstrate that the full pipeline - data loading, augmentation, the YOLOv8 training
loop, loss computation and validation - executes correctly end-to-end, and that the
loss decreases within that single epoch (i.e. the model is learning something), but
the resulting weights are not a meaningfully trained detector. plots=False is set so
that Ultralytics does not save training/validation batch mosaics, since those images
would embed actual dataset slices and this dataset must not be redistributed. As a
direct consequence, Tasks 4 and 5 below (evaluation and qualitative results against a
real trained model) could not be produced for this submission - see the Limitations
discussion in the report for the full explanation and what a completed run would
involve.

In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=1,  # proof-of-concept only, see note above
    imgsz=512,
    batch=16,
    project="../runs",
    name="osteotomy_yolov8n_poc",
    seed=42,
    plots=False,  # avoid saving mosaics/prediction images that embed dataset content
)

Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\dataset\yolo\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=osteotomy_yolov8n_poc, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 2/4 3.5s/it 3.6s<7.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 75% ━━━━━━━━━─── 3/4 2.7s/it 5.3s<2.7s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.7s/it 6.9s

                   all        123        180   8.13e-05     0.0167   2.12e-06   2.51e-07


Speed: 0.8ms preprocess, 41.5ms inference, 0.0ms loss, 11.2ms postprocess per image
